In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing import sequence


Data Load

In [ ]:
# # Read TXT file line by line
# sequences = []
# with open("all_analysis_data.txt", "r") as f:
#     for line in f:
#         tokens = line.strip().lower().split()  # lowercase & split
#         sequences.append(tokens)

# print(f"Total samples: {len(sequences)}")
# print(f"Example sequence: {sequences[0][:20]}")  # first 20 tokens of first sample

In [ ]:
# def collapse_duplicates(seq):
#     if not seq:
#         return []
#     collapsed = [seq[0]]
#     for token in seq[1:]:
#         if token != collapsed[-1]:
#             collapsed.append(token)
#     return collapsed

# # Apply to all sequences
# sequences = [collapse_duplicates(seq) for seq in sequences]

# print(f"Example after collapsing duplicates: {sequences[0][:20]}")


In [ ]:
# import csv

# with open("all_analysis_data.csv", "w", newline="") as f:
#     writer = csv.writer(f)

#     for seq in sequences:
#         row_text = " ".join(seq)   # convert list -> single string
#         writer.writerow([row_text])  # write one-column row


In [ ]:
labels = pd.read_csv("labels.csv", header=None)
df = pd.read_csv("all_analysis_data.csv", header=None)
df.columns = ["API_Calls"]
df["Labels"] = labels
df.head()

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

In [ ]:
df.Labels.value_counts()

In [ ]:
df.to_csv("preprocessed_dataset.csv", index=False)

In [ ]:
df = pd.read_csv("preprocessed_dataset.csv")
max_words = 800
max_len = 200

X = df.API_Calls
y = df.Labels.astype("category").cat.codes

tok = Tokenizer(num_words=max_words)
tok.fit_on_texts(X)
print("unique tokens:",len(tok.word_index))
X = tok.texts_to_sequences(X.values)
X = sequence.pad_sequences(X, maxlen=max_len)
print('Shape of data tensor:', X.shape)


In [ ]:
y

In [ ]:
df_save = pd.DataFrame(X)  # integer token sequences
df_save['Label'] = y           # integer labels

df_save.to_csv("malware_sequences_int.csv", index=False)

In [ ]:
df_save

In [ ]:
df = pd.read_csv("malware_sequences_int.csv")
X = df.drop(columns="Label")
y = df.Label

X_train, X_temp, Y_train, Y_temp = train_test_split(
    X, y, test_size=0.25, random_state=42,stratify=y
)

X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp, Y_temp, test_size=0.6, random_state=42, stratify=Y_temp
)

# train -> 75%, Val -> 10%, Test -> 15%

In [ ]:
num_classes = len(np.unique(Y_train))

In [ ]:
# Y_train_oh = to_categorical(Y_train, num_classes)
# Y_test_oh = to_categorical(Y_test, num_classes)
# Y_val_oh = to_categorical(Y_val, num_classes)

In [ ]:
max_words = 800
max_len = 200

def malware_model(act_fuc="relu"):
    model = Sequential()
    model.add(Embedding(max_words, 128, input_length = max_len))
    model.add(SpatialDropout1D(0.1))
    model.add(LSTM(64, dropout=0.1, recurrent_dropout=0.1, return_sequences=True))
    model.add(LSTM(64, dropout=0.1))
    model.add(Dense(128, activation=act_fuc))
    model.add(Dropout(0.1))
    model.add(Dense(128, activation=act_fuc))
    model.add(Dropout(0.1))
    model.add(Dense(8, name="out_layer", activation="softmax"))

    return model

    

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(Y_train),
    y=Y_train
)

class_weights = dict(enumerate(class_weights))


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

model = malware_model()
model.compile( loss="sparse_categorical_crossentropy", optimizer="Adam", metrics=["accuracy"] )
model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=100,
    batch_size=64,
    callbacks=[es],
    class_weight=class_weights
)


In [ ]:
Y_test_pred = model.predict(X_test)

y_pred_classes = np.argmax(Y_test_pred, axis=1)

print(confusion_matrix(Y_test, y_pred_classes))
print(classification_report(Y_test, y_pred_classes))
